In [ ]:
import torch
from diffusers import WanImageToVideoPipeline
from diffusers.utils import load_image, export_to_video

#model_id = "Wan-AI/Wan2.2-I2V-A14B-Diffusers"
model_id = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"

pipe = WanImageToVideoPipeline.from_pretrained(model_id, torch_dtype=torch.bfloat16)
pipe.to("cuda")  # 24GB 이상 권장, 80GB면 720p 풀옵

In [ ]:
from PIL import Image

url = "image.png"  # 로컬 파일 혹은 URL
image = load_image(url).convert("RGB")

# 1) 720p를 위한 타깃 해상도
target_w, target_h = 1280, 720

# 2) 비율 유지한 채로 먼저 리사이즈 (둘 다 타깃보다 크거나 같도록)
w, h = image.size
scale = max(target_w / w, target_h / h)  # 둘 중 더 큰 스케일 사용
new_w = int(round(w * scale))
new_h = int(round(h * scale))
image = image.resize((new_w, new_h), Image.BICUBIC)

# 3) 센터 크롭으로 정확히 1280×720 만들기
left = (new_w - target_w) // 2
top = (new_h - target_h) // 2
right = left + target_w
bottom = top + target_h
image = image.crop((left, top, right, bottom))

out = pipe(
    prompt="sexy girl",
    #image=image,
    num_frames=16*5,
    generator=torch.manual_seed(0),
)

frames = out.frames[0]
export_to_video(frames, "wan22_i2v.mp4", fps=16)


NameError: name 'load_image' is not defined